# Самостоятельная работа: задачи 1–9
По рабочей тетради 7-8. Цапков М.А. ЭПМО-01-26

Язык — Python 3. Используется только стандартная библиотека.


## 1. Проверка ссылок односвязного списка

**Условие.** Обнаружить проскоки, перескоки и некорректные ссылки без множества, хеш-таблицы и хранения посещенных узлов. Ссылки в тестах задаются вручную.

### 1. Алгоритм и структуры данных

Узел содержит value и единственное поле next. Нужен эталон логического порядка: каждому узлу при создании присваиваем неизменяемый номер position от 0 до N−1. Проверка получает head и ожидаемое число узлов N. По очереди требует позиции 0, 1, …, N−1 и затем None; проверяет тип next. Возврат назад нарушает ожидаемый номер, пропуск — тоже. Число шагов ограничено N.

### 2. Обоснование выбора

Без эталона задача в общем случае неразрешима: цепочка A→C→None сама по себе корректна, и узнать о пропущенном B невозможно. Поэтому явно принимаем уникальность и неизменность номеров исходных узлов. position — метаданные порядка, а не отметка посещения. Значение value не обязано быть уникальным. Проверка не создает вспомогательный список узлов.

### 3. Вычислительная сложность и масштабирование

Время O(N), дополнительная память O(1); метаданные исходных узлов занимают O(N). Цикл завершается ошибкой на первом неверном номере, отдельное хранение посещений не требуется.

### 4. Программная реализация


In [1]:
class Node:
    __slots__ = ("value", "next", "_position")

    def __init__(self, value, position=None):
        self.value = value
        self.next = None
        self._position = position

    @property
    def position(self):
        return self._position


def check_links(head, expected_count):
    if expected_count < 0:
        return "error: отрицательное число узлов"
    current = head
    for expected in range(expected_count):
        if not isinstance(current, Node):
            return "error: отсутствует узел или неверный тип ссылки"
        if current.position != expected:
            return "error: проскок или перескок"
        if current.next is not None and not isinstance(current.next, Node):
            return "error: неверный тип next"
        current = current.next
    return "ok" if current is None else "error: лишняя ссылка или цикл"


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [2]:
a, b, d = Node("A", 0), Node("B", 1), Node("C", 2)
a.next = b
b.next = d
d.next = None
assert check_links(a, 3) == "ok"
print("Корректный список:", check_links(a, 3))
a.next = d
assert check_links(a, 3) != "ok"
print("Пропущен B:", check_links(a, 3))
a.next = b
d.next = b
assert check_links(a, 3) != "ok"
print("Цикл C→B:", check_links(a, 3))
d.next = None
b.next = a
assert check_links(a, 3) != "ok"
print("Возврат B→A:", check_links(a, 3))
b.next = 42
assert check_links(a, 3) != "ok"
print("Некорректный next:", check_links(a, 3))
b.next = None
assert check_links(a, 3) != "ok"
assert check_links(None, 0) == "ok"
assert check_links(None, 1) != "ok"
a.next = a
assert check_links(a, 1) != "ok"
a.next = None
assert check_links(a, 1) == "ok"


Корректный список: ok
Пропущен B: error: проскок или перескок
Цикл C→B: error: лишняя ссылка или цикл
Возврат B→A: error: проскок или перескок
Некорректный next: error: неверный тип next


### 6.* Вывод и возможная оптимизация

Проверка обнаруживает нарушения ссылок при доверенных номерах и известном количестве узлов. Если эталонных сведений нет, алгоритм Флойда выявит циклы, но не позволит обнаружить ациклический пропуск узла.


## 2. Кольцевой двусвязный список

**Условие.** Узел value/next/prev; добавление в конец, удаление первого вхождения, вывод от head до head. Все изменения ссылок — внутри функций.

### 1. Алгоритм и структуры данных

Для непустого кольца хвост равен head.prev. Добавление связывает новый узел между хвостом и head. Удаление сшивает соседей; при удалении единственного узла возвращает None. Проверка кольца контролирует взаимность ссылок и число узлов.

### 2. Обоснование выбора

Две ссылки позволяют удалить найденный узел без поиска предшественника; кольцо дает доступ к хвосту через head.prev.

### 3. Вычислительная сложность и масштабирование

Добавление O(1), поиск и удаление по значению O(N), вывод и проверка O(N). Хранение O(N), дополнительная память изменения O(1); список значений для вывода O(N).

### 4. Программная реализация


In [3]:
class DNode:
    def __init__(self, value):
        self.value = value
        self.next = None
        self.prev = None


def ring_append(head, value):
    node = DNode(value)
    if head is None:
        node.next = node.prev = node
        return node
    tail = head.prev
    node.prev, node.next = tail, head
    tail.next = node
    head.prev = node
    return head


def ring_remove(head, value):
    if head is None:
        return None
    current = head
    while True:
        if current.value == value:
            if current.next is current:
                current.next = current.prev = None
                return None
            current.prev.next = current.next
            current.next.prev = current.prev
            if current is head:
                head = current.next
            current.next = current.prev = None
            return head
        current = current.next
        if current is head:
            return head


def ring_values(head):
    if head is None:
        return []
    result = []
    current = head
    while True:
        result.append(current.value)
        current = current.next
        if current is head:
            return result


def ring_is_valid(head, expected_count):
    if head is None:
        return expected_count == 0
    current = head
    for i in range(expected_count):
        if current is None or current.next is None or current.prev is None:
            return False
        if current.next.prev is not current or current.prev.next is not current:
            return False
        if i > 0 and current is head:
            return False
        current = current.next
    return expected_count > 0 and current is head


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [4]:
head = None
assert ring_is_valid(head, 0)
assert ring_remove(head, 10) is None
for size, value in enumerate([1, 2, 2, 3], 1):
    head = ring_append(head, value)
    assert ring_is_valid(head, size)
print("Начальное кольцо:", ring_values(head))
for value, expected in [(2, [1, 2, 3]), (1, [2, 3]), (3, [2]), (99, [2]), (2, [])]:
    head = ring_remove(head, value)
    assert ring_values(head) == expected
    assert ring_is_valid(head, len(expected))
    print("Удаление", value, "→", ring_values(head))
assert head is None


Начальное кольцо: [1, 2, 2, 3]
Удаление 2 → [1, 2, 3]
Удаление 1 → [2, 3]
Удаление 3 → [2]
Удаление 99 → [2]
Удаление 2 → []


### 6.* Вывод и возможная оптимизация

Кольцо удобно для циклического обхода; условие остановки — возврат в head. Индекс по значению может ускорить поиск, но требует поддержки повторяющихся значений.


## 3. Расширенный двусвязный список

**Условие.** Добавить в конец, вставить перед первым совпадением, удалить все совпадения, посчитать узлы, вывести в обе стороны, проверить палиндром.

### 1. Алгоритм и структуры данных

Храним head и tail; используем DNode из задания 2. Каждая операция — отдельная функция. При изменениях обновляем ссылки соседей и границы. Для палиндрома сравниваем значения с двух концов до встречи указателей.

### 2. Обоснование выбора

Двусвязность позволяет двигаться в обе стороны без копирования и обновлять соседей за O(1) после нахождения узла.

### 3. Вычислительная сложность и масштабирование

Добавление O(1), вставка с поиском O(N), удаление всех O(N), подсчет и вывод O(N), палиндром O(N). Хранение O(N); палиндром и изменение ссылок требуют O(1) дополнительной памяти, вывод в список — O(N).

### 4. Программная реализация


In [5]:
class DoublyList:
    def __init__(self):
        self.head = None
        self.tail = None


def append_value(linked, value):
    node = DNode(value)
    node.prev = linked.tail
    if linked.tail is None:
        linked.head = node
    else:
        linked.tail.next = node
    linked.tail = node


def insert_before(linked, target, value):
    current = linked.head
    while current is not None and current.value != target:
        current = current.next
    if current is None:
        return False
    node = DNode(value)
    node.prev, node.next = current.prev, current
    if current.prev is None:
        linked.head = node
    else:
        current.prev.next = node
    current.prev = node
    return True


def remove_all(linked, value):
    current = linked.head
    removed = 0
    while current is not None:
        following = current.next
        if current.value == value:
            if current.prev is None:
                linked.head = current.next
            else:
                current.prev.next = current.next
            if current.next is None:
                linked.tail = current.prev
            else:
                current.next.prev = current.prev
            current.next = current.prev = None
            removed += 1
        current = following
    return removed


def count_nodes(linked):
    count = 0
    current = linked.head
    while current is not None:
        count += 1
        current = current.next
    return count


def forward_values(linked):
    result = []
    current = linked.head
    while current is not None:
        result.append(current.value)
        current = current.next
    return result


def backward_values(linked):
    result = []
    current = linked.tail
    while current is not None:
        result.append(current.value)
        current = current.prev
    return result


def is_palindrome(linked):
    left, right = linked.head, linked.tail
    while left is not right and left is not None and right is not None:
        if left.value != right.value:
            return False
        if left.next is right:
            break
        left, right = left.next, right.prev
    return True


def doubly_is_valid(linked):
    previous = None
    current = linked.head
    while current is not None:
        if current.prev is not previous:
            return False
        previous, current = current, current.next
    return previous is linked.tail


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [6]:
linked = DoublyList()
assert is_palindrome(linked) and count_nodes(linked) == 0
assert not insert_before(linked, 9, 0)
assert remove_all(linked, 1) == 0
for value in [1, 2, 2, 1]:
    append_value(linked, value)
    assert doubly_is_valid(linked)
assert is_palindrome(linked)
assert count_nodes(linked) == 4
assert insert_before(linked, 2, 9)
assert forward_values(linked) == [1, 9, 2, 2, 1]
assert backward_values(linked) == [1, 2, 2, 9, 1]
assert not is_palindrome(linked)
assert doubly_is_valid(linked)
print("Прямой порядок:", forward_values(linked))
print("Обратный порядок:", backward_values(linked))
assert remove_all(linked, 2) == 2
assert doubly_is_valid(linked)
assert remove_all(linked, 1) == 2
assert forward_values(linked) == [9] and is_palindrome(linked)
assert doubly_is_valid(linked)
assert insert_before(linked, 9, 8)
assert forward_values(linked) == [8, 9] and doubly_is_valid(linked)
assert remove_all(linked, 9) == 1
assert linked.head is linked.tail
assert remove_all(linked, 8) == 1
assert linked.head is linked.tail is None
assert doubly_is_valid(linked)
for sample in [[1], [1, 2, 1], [1, 2], [4, 4, 4]]:
    example = DoublyList()
    for value in sample:
        append_value(example, value)
    assert is_palindrome(example) == (sample == sample[::-1])
    remove_all(example, 4)
    assert doubly_is_valid(example)
print("Проверки пустого, одиночного списка и границ пройдены")


Прямой порядок: [1, 9, 2, 2, 1]
Обратный порядок: [1, 2, 2, 9, 1]
Проверки пустого, одиночного списка и границ пройдены


### 6.* Вывод и возможная оптимизация

Операции не требуют копии списка. Для частого подсчета можно хранить размер и обновлять его при изменениях; это даст O(1) вместо обхода.


## 4. Регистратор заданий

**Условие.** Добавить [101,102,103,104], удалить 102, добавить 105; обслужить FIFO на связном списке head/tail.

### 1. Алгоритм и структуры данных

Добавляем узел через tail; при выдаче снимаем head. Удаление по ID выполняет обход с предыдущим узлом и при необходимости обновляет tail.

### 2. Обоснование выбора

Ссылка tail устраняет полный проход при каждой вставке, head дает выдачу в порядке FIFO.

### 3. Вычислительная сложность и масштабирование

Добавление и выдача O(1), удаление по ID O(N); память O(N), дополнительная память операций O(1). После изменений 4 задания: 101,103,104,105. Для построения N узлов без tail пришлось бы выполнить (N−1)(N−2)/2 переходов next при обычном поиске хвоста.

### 4. Программная реализация


In [7]:
class LinkedQueue:
    def __init__(self):
        self.head = None
        self.tail = None

    def push(self, value):
        node = Node(value)
        if self.tail is None:
            self.head = node
        else:
            self.tail.next = node
        self.tail = node
        return node

    def pop(self):
        if self.head is None:
            raise IndexError("Пустая очередь")
        node = self.head
        self.head = node.next
        node.next = None
        if self.head is None:
            self.tail = None
        return node.value

    def remove(self, value):
        previous = None
        current = self.head
        while current is not None:
            if current.value == value:
                if previous is None:
                    self.head = current.next
                else:
                    previous.next = current.next
                if current is self.tail:
                    self.tail = previous
                current.next = None
                return True
            previous, current = current, current.next
        return False


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [8]:
queue = LinkedQueue()
for identifier in [101, 102, 103, 104]:
    queue.push(identifier)
assert queue.remove(102)
queue.push(105)
assert queue.head.value == 101 and queue.tail.value == 105
order = []
while queue.head is not None:
    order.append(queue.pop())
assert order == [101, 103, 104, 105]
assert queue.tail is None
assert not queue.remove(999)
queue.push(1)
assert queue.remove(1) and queue.head is queue.tail is None
queue.push(1)
queue.push(2)
assert queue.remove(2) and queue.tail is queue.head
assert queue.remove(1) and queue.tail is None
try:
    queue.pop()
except IndexError:
    pass
else:
    raise AssertionError("Ожидалась ошибка пустой очереди")
print("Обслуживание:", order)


Обслуживание: [101, 103, 104, 105]


### 6.* Вывод и возможная оптимизация

Для быстрого удаления по ID нужны словарь ID→узел и двусвязность: ссылка на узел односвязного списка сама по себе не дает его предшественника.


## 5. Обнаружение и устранение цикла

**Условие.** В цепочке 1→2→3→4→5 ошибочная ссылка 5→3. Найти цикл, удалить его, развернуть список.

### 1. Алгоритм и структуры данных

Алгоритм Флойда находит встречу быстрого и медленного указателей. Переносим один к head и двигаем оба по одному до входа в цикл. Обходим цикл до узла, ссылающегося на вход, и обрываем ссылку. Разворот выполняем тремя указателями.

### 2. Обоснование выбора

Флойд обнаруживает цикл без множества и изменения значений узлов. После восстановления линейности разворот безопасно завершается.

### 3. Вычислительная сложность и масштабирование

Каждый этап O(N) времени и O(1) дополнительной памяти. Для примера вход — 3, длина цикла 3, линейный префикс 2 узла; обрывается 5→3. После разворота: 5,4,3,2,1.

### 4. Программная реализация


In [9]:
def cycle_entry(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            first = head
            while first is not slow:
                first = first.next
                slow = slow.next
            return first
    return None


def remove_cycle(head):
    entry = cycle_entry(head)
    if entry is None:
        return None
    last = entry
    while last.next is not entry:
        last = last.next
    last.next = None
    return entry.value


def reverse_list(head):
    previous = None
    current = head
    while current is not None:
        following = current.next
        current.next = previous
        previous, current = current, following
    return previous


def list_values(head):
    result = []
    while head is not None:
        result.append(head.value)
        head = head.next
    return result


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [10]:
nodes = [Node(i) for i in range(1, 6)]
for left, right in zip(nodes, nodes[1:]):
    left.next = right
nodes[-1].next = nodes[2]
assert cycle_entry(nodes[0]) is nodes[2]
print("Вход в цикл:", remove_cycle(nodes[0]))
assert cycle_entry(nodes[0]) is None
assert list_values(nodes[0]) == [1, 2, 3, 4, 5]
head = reverse_list(nodes[0])
assert list_values(head) == [5, 4, 3, 2, 1]
print("После разворота:", list_values(head))
assert remove_cycle(None) is None
assert reverse_list(None) is None
single = Node(1)
single.next = single
assert remove_cycle(single) == 1 and single.next is None
assert reverse_list(single) is single
assert remove_cycle(single) is None
x, y = Node(1), Node(2)
x.next, y.next = y, x
assert remove_cycle(x) == 1
assert list_values(x) == [1, 2]


Вход в цикл: 3
После разворота: [5, 4, 3, 2, 1]


### 6.* Вывод и возможная оптимизация

Флойд подходит для проверки циклов в больших цепочках с постоянной памятью. Для профилактики нужны корректные операции обновления ссылок.


## 6. Журнал отмены и скобки

**Условие.** Команды [A,B,C,D], отменить две; проверить ([{}]).

### 1. Алгоритм и структуры данных

Список Python используем как стек: append добавляет команду, pop отменяет последнюю. Второй стек хранит открывающие скобки и проверяет соответствие закрывающих.

### 2. Обоснование выбора

Обе задачи требуют LIFO: отменяется последняя команда и первой закрывается последняя открытая скобка.

### 3. Вычислительная сложность и масштабирование

Добавление и отмена амортизированно O(1); проверка L символов O(L) времени и O(L) памяти в худшем случае. После двух отмен остаются A,B; отменены D,C. Максимальная вложенность заданной строки 3.

### 4. Программная реализация


In [11]:
def undo(history, count):
    if count < 0 or count > len(history):
        raise ValueError("Некорректное число отмен")
    cancelled = []
    for _ in range(count):
        cancelled.append(history.pop())
    return cancelled


def brackets_valid(text):
    opening = []
    pairs = {")": "(", "]": "[", "}": "{"}
    for symbol in text:
        if symbol in "([{":
            opening.append(symbol)
        elif symbol in pairs:
            if not opening or opening.pop() != pairs[symbol]:
                return False
    return not opening


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [12]:
history = list("ABCD")
assert undo(history, 2) == ["D", "C"]
assert history == ["A", "B"]
assert undo(history, 0) == []
assert brackets_valid("([{}])")
for sample in ["", "()[]{}", "a(b)c"]:
    assert brackets_valid(sample)
for sample in ["([)]", ")(", "((", "{]"]:
    assert not brackets_valid(sample)
print("Оставшиеся команды:", history)
print("Скобки ([{}]):", brackets_valid("([{}])"))


Оставшиеся команды: ['A', 'B']
Скобки ([{}]): True


### 6.* Вывод и возможная оптимизация

Стек естественно отражает вложенность. Для настоящего редактора потребуется хранить обратное действие каждой команды, а не только ее имя.


## 7. FIFO, срочность и приоритеты

**Условие.** FIFO: 201,202,203; срочный 299 ставится в начало. Приоритеты (2,201),(1,202),(3,203).

### 1. Алгоритм и структуры данных

Сравниваем три самостоятельных порядка: исходную FIFO, дек с appendleft для 299 и min-heap для заданных приоритетов. Меньшее число означает более высокий приоритет; у 299 числовой приоритет не задан, поэтому в heap его не включаем.

### 2. Обоснование выбора

FIFO сохраняет поступление, дек дает срочную вставку, куча выбирает минимальный приоритет.

### 3. Вычислительная сложность и масштабирование

FIFO и дек: O(1) на операцию; куча: построение O(N), извлечение O(log N). Память O(N). Порядки: 201,202,203; 299,201,202,203; 202,201,203.

### 4. Программная реализация


In [13]:
from collections import deque
import heapq


def compare_orders(ids, urgent, priorities):
    fifo = deque(ids)
    urgent_queue = deque(ids)
    urgent_queue.appendleft(urgent)
    heap = list(priorities)
    heapq.heapify(heap)
    priority_order = []
    while heap:
        priority, identifier = heapq.heappop(heap)
        priority_order.append(identifier)
    return list(fifo), list(urgent_queue), priority_order


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [14]:
orders = compare_orders([201, 202, 203], 299, [(2, 201), (1, 202), (3, 203)])
assert orders == ([201, 202, 203], [299, 201, 202, 203], [202, 201, 203])
for name, order in zip(["FIFO", "Срочный в начало", "Приоритетная очередь"], orders):
    print(name, order)
assert compare_orders([], 299, []) == ([], [299], [])


FIFO [201, 202, 203]
Срочный в начало [299, 201, 202, 203]
Приоритетная очередь [202, 201, 203]


### 6.* Вывод и возможная оптимизация

Выбор зависит от политики обслуживания. Для равных приоритетов и сохранения порядка поступления можно добавить счетчик в элементы кучи; непрерывные срочные запросы требуют защиты обычных от голодания.


## 8. Структура при частых вставках

**Условие.** 20 000 узлов, 5 000 добавлений в конец/с, 2 000 запросов по ID/с. Сравнить head, head/tail и индекс.

### 1. Алгоритм и структуры данных

Для варианта с head хвост ищем обходом. Для head/tail используем LinkedQueue из задания 4. Индексированный вариант добавляет dict ID→узел, поддерживаемый при каждой вставке и выдаче. Предполагаем уникальные ID.

### 2. Обоснование выбора

При частых добавлениях нужен tail, при частых запросах — словарь. Одно улучшение не устраняет другое узкое место.

### 3. Вычислительная сложность и масштабирование

Только head: вставка и поиск O(N), память O(N). Head/tail: вставка O(1), поиск O(N), память O(N). С индексом: оба в среднем O(1), дополнительный индекс O(N). При фиксированном N=20 000: поиск хвоста дает 5 000×19 999=99 995 000 переходов/с, поиск ID — до 40 000 000 проверок/с. С tail остаются 5 000 постоянных вставок и до 40 млн проверок; с индексом — около 7 000 операций словаря/очереди. Если за секунду только добавлять и не удалять, сумма переходов для 5 000 вставок составит 112 492 500, поскольку список растет.

### 4. Программная реализация


In [15]:
def find_by_id(head, identifier):
    while head is not None:
        if head.value == identifier:
            return head
        head = head.next
    return None


def append_head_only(head, value):
    node = Node(value)
    if head is None:
        return node
    current = head
    while current.next is not None:
        current = current.next
    current.next = node
    return head


class IndexedQueue(LinkedQueue):
    def __init__(self):
        super().__init__()
        self.by_id = {}

    def push(self, identifier):
        if identifier in self.by_id:
            return False
        node = super().push(identifier)
        self.by_id[identifier] = node
        return True

    def pop(self):
        identifier = super().pop()
        del self.by_id[identifier]
        return identifier

    def remove(self, identifier):
        if identifier not in self.by_id:
            return False
        # Односвязная очередь все еще требует поиска предшественника.
        super().remove(identifier)
        del self.by_id[identifier]
        return True

    def find(self, identifier):
        return self.by_id.get(identifier)


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [16]:
head = None
queue = LinkedQueue()
indexed = IndexedQueue()
for identifier in [10, 20, 30]:
    head = append_head_only(head, identifier)
    queue.push(identifier)
    assert indexed.push(identifier)
assert list_values(head) == list_values(queue.head) == list_values(indexed.head) == [10, 20, 30]
assert find_by_id(head, 20).value == indexed.find(20).value == 20
assert find_by_id(queue.head, 99) is indexed.find(99) is None
assert not indexed.push(20)
assert indexed.pop() == 10 and indexed.find(10) is None
assert indexed.remove(30) and indexed.find(30) is None
assert indexed.pop() == 20 and indexed.head is indexed.tail is None
assert indexed.by_id == {}
print("Поиск 20 во всех трех вариантах успешен")
n, inserts, queries = 20_000, 5_000, 2_000
print("Head, переходы при вставках (фиксированный N):", inserts * (n - 1))
print("Линейный поиск, проверок/с до:", n * queries)
print("Индексированный вариант, операций порядка:", inserts + queries)
print("Переходы при росте списка за секунду:", inserts * (n - 1) + inserts * (inserts - 1) // 2)


Поиск 20 во всех трех вариантах успешен
Head, переходы при вставках (фиксированный N): 99995000
Линейный поиск, проверок/с до: 40000000
Индексированный вариант, операций порядка: 7000
Переходы при росте списка за секунду: 112492500


### 6.* Вывод и возможная оптимизация

Head/tail со словарем подходит заданной нагрузке, но увеличивает память. Удаление произвольного узла в этой односвязной версии остается O(N); для ускорения удаления нужен двусвязный список.


## 9. Поток с повторной постановкой

**Условие.** События [(501,ok),(502,error),(503,ok),(504,error)]. Ошибочные обработать повторно один раз с постановкой в начало.

### 1. Алгоритм и структуры данных

Дек хранит события, словарь считает попытки. При error на первой попытке кладем событие в начало. На второй попытке больше не возвращаем его в очередь. Сохраняем конечный статус failed, если повтор тоже завершился ошибкой; условие не задает смену результата.

### 2. Обоснование выбора

appendleft позволяет немедленно повторить ошибочное событие, а ограничение попыток гарантирует завершение. Первые попытки сохраняют FIFO.

### 3. Вычислительная сложность и масштабирование

В среднем O(N+E) времени для E ошибочных событий, O(N) памяти. Здесь N=4, E=2, всего 6 попыток: 501,502,502,503,504,504.

### 4. Программная реализация


In [17]:
def process_stream(events):
    queue = deque(events)
    attempts = {}
    final_states = {}
    order = []
    while queue:
        identifier, status = queue.popleft()
        attempts[identifier] = attempts.get(identifier, 0) + 1
        order.append((identifier, attempts[identifier]))
        if status == "error" and attempts[identifier] == 1:
            queue.appendleft((identifier, status))
            final_states[identifier] = "retry"
        else:
            final_states[identifier] = "failed" if status == "error" else "done"
    return order, attempts, final_states


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [18]:
events = [(501, "ok"), (502, "error"), (503, "ok"), (504, "error")]
order, attempts, states = process_stream(events)
assert order == [(501, 1), (502, 1), (502, 2), (503, 1), (504, 1), (504, 2)]
assert [identifier for identifier, attempt in order if attempt == 1] == [501, 502, 503, 504]
assert attempts == {501: 1, 502: 2, 503: 1, 504: 2}
assert states == {501: "done", 502: "failed", 503: "done", 504: "failed"}
assert process_stream([]) == ([], {}, {})
print("Порядок (ID, попытка):", order)
print("Число попыток:", attempts)
print("Итоговые состояния:", states)


Порядок (ID, попытка): [(501, 1), (502, 1), (502, 2), (503, 1), (504, 1), (504, 2)]
Число попыток: {501: 1, 502: 2, 503: 1, 504: 2}
Итоговые состояния: {501: 'done', 502: 'failed', 503: 'done', 504: 'failed'}


### 6.* Вывод и возможная оптимизация

Повторная постановка ограничена, поэтому очередь завершается. Для реального сервиса можно добавить задержку между попытками и отдельную очередь окончательно неуспешных событий.
